In [ ]:
# Llama-3.2 is gated. Store a Kaggle secret named HF_TOKEN.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login: OK')
except Exception as exc:
    print('HF login was not completed automatically:', exc)
    print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
# TODO for Question 4
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""
layers_to_analyze = [0, 7, 15]

for k in layers_to_analyze:
    print('Layer:', k)
  
    # Extract query projection layer where layer = k
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()
    
    # Doing SVD on the layer 
    U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)
    
    # Compute Cumulative energy
    energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
    
    # Compute the minimum target rank at least 50% cumulative energy
    min_50_rank = (energy_ratios >= 50.0).nonzero(as_tuple=True)[0][0].item() + 1
    print('Minimum Target Rank at least 50% cumulative energy =', min_50_rank)
    
    # visualizations: bar chart & line chart
    s_200 = S[:200].cpu().numpy()
    energy_200 = energy_ratios[:200].cpu().numpy()
    
    # bar chart for Singular Value Magnitude
    plt.figure(figsize=(6, 3))
    plt.bar(range(1, 201), s_200, color='steelblue', width=0.8)
    plt.grid()
    plt.xlabel('Singular Value Index')
    plt.ylabel('Singular Value  Magnitude')
    plt.show()
    
    # line chart for Cumulative Energy
    plt.figure(figsize=(6, 3))
    plt.plot(range(1, 201), energy_200, color='steelblue', linewidth=2)
    plt.grid()
    plt.xlabel('Target Rank')
    plt.ylabel('Cumulative Energy (%)')
    plt.show()




